# 22 - Word2Vec + Self-Attention (window=32)

Goal: Usar **self-attention manual** con una ventana de contexto de **32 palabras**.

**Problema en 19:** RNN con window=64 falló rotundamente (10.5% accuracy) por vanishing gradient.

**Hipótesis:** Self-attention NO sufre vanishing gradient (conexión directa entre cualquier par),
por lo que debería manejar window=32 sin problemas.

Run with: conda activate tfenv

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
class Word2VecLoader:
    def __init__(self, path='myWord2Vec/v2/'):
        target_embeddings = np.load(path + 'target_embeddings.npy')
        context_embeddings = np.load(path + 'context_embeddings.npy')
        text_vocab = np.load(path + 'text_vocab.npy', allow_pickle=True).item()

        self.target_embeddings = target_embeddings
        self.context_embeddings = context_embeddings
        self.final_embeddings = (target_embeddings + context_embeddings) / 2
        self.text_vocab = text_vocab
        self.idx_to_word = {idx: word for word, idx in text_vocab.items()}
        self.embedding_dim = target_embeddings.shape[1]

        self.embedding_layer = layers.Embedding(
            input_dim=target_embeddings.shape[0],
            output_dim=target_embeddings.shape[1],
            weights=[target_embeddings],
            trainable=False,
            name='pretrained_embedding'
        )

        print('Embeddings cargados:', target_embeddings.shape, context_embeddings.shape)
        print('Vocabulario cargado:', len(text_vocab))

    def encode(self, words):
        return [self.text_vocab[w] for w in words if w in self.text_vocab]

    def decode(self, token_id):
        return self.idx_to_word.get(int(token_id), '<unk>')

loader = Word2VecLoader()

In [ ]:
WINDOW = 32  # potencia de 2

print('Loading gaianet/london dataset...')
ds = load_dataset('gaianet/london', split='train')
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:50000]
full_text = ' '.join(texts[:50000])

words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2 and w in loader.text_vocab]
print(f'Total words used: {len(words)}')
print(f'Sample: {words[:20]}')

# step=1 para maximizar datos (a diferencia de notebook 19 que usaba step=5 y perdió datos)
def create_sequences(words, window=WINDOW, step=1):
    X, y = [], []
    for i in range(0, len(words) - window, step):
        context = words[i:i + window]
        target = words[i + window]
        if all(w in loader.text_vocab for w in context + [target]):
            X.append([loader.text_vocab[w] for w in context])
            y.append(loader.text_vocab[target])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X, y = create_sequences(words, window=WINDOW)
print(f'Sequences created: {len(X)}')
print(f'Input shape: {X.shape}, Target shape: {y.shape}')

if len(X) == 0:
    raise ValueError('No se pudieron crear secuencias.')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
# Positional Encoding para 32 posiciones
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angles = pos * angle_rates
    pe = np.zeros_like(angles)
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(pe[np.newaxis, :, :], tf.float32)

pe = positional_encoding(WINDOW, 64)[0]

plt.figure(figsize=(10, 4))
plt.imshow(pe.numpy(), cmap='RdBu', aspect='auto')
plt.colorbar(label='Valor')
plt.xlabel('Dimensión del embedding')
plt.ylabel('Posición en la secuencia')
plt.title(f'Positional Encoding ({WINDOW} posiciones, 64 dimensiones)')
plt.tight_layout()
plt.show()

In [ ]:
# Misma ManualSelfAttention que notebook 21, pero con window=32
class ManualSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads=4, dropout_rate=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.Wq = layers.Dense(embed_dim, use_bias=False, name='Wq')
        self.Wk = layers.Dense(embed_dim, use_bias=False, name='Wk')
        self.Wv = layers.Dense(embed_dim, use_bias=False, name='Wv')
        self.Wo = layers.Dense(embed_dim, use_bias=False, name='Wo')
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, return_attention=False):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = tf.reshape(Q, (B, T, self.num_heads, self.head_dim))
        K = tf.reshape(K, (B, T, self.num_heads, self.head_dim))
        V = tf.reshape(V, (B, T, self.num_heads, self.head_dim))

        Q = tf.transpose(Q, [0, 2, 1, 3])
        K = tf.transpose(K, [0, 2, 1, 3])
        V = tf.transpose(V, [0, 2, 1, 3])

        scale = tf.sqrt(tf.cast(self.head_dim, tf.float32))
        scores = tf.matmul(Q, K, transpose_b=True) / scale
        att_weights = tf.nn.softmax(scores, axis=-1)
        att_weights = self.dropout(att_weights)

        context = tf.matmul(att_weights, V)

        context = tf.transpose(context, [0, 2, 1, 3])
        context = tf.reshape(context, (B, T, self.num_heads * self.head_dim))
        out = self.Wo(context)

        if return_attention:
            att_avg = tf.reduce_mean(att_weights, axis=1)
            return out, att_avg
        return out


class SelfAttentionWordPredictor(keras.Model):
    def __init__(self, embedding_layer, vocab_size, embed_dim, seq_len, num_heads=4):
        super().__init__()
        self.seq_len = seq_len
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.embedding = embedding_layer
        self.pos_encoding = positional_encoding(seq_len, embed_dim)

        self.self_attn = ManualSelfAttention(embed_dim, num_heads, dropout_rate=0.1)
        self.layer_norm = layers.LayerNormalization(epsilon=1e-6)
        self.pool = layers.GlobalAveragePooling1D()
        self.out = layers.Dense(vocab_size, activation='softmax')

    def call(self, inputs, return_attention=False):
        x = self.embedding(inputs)
        x = x + self.pos_encoding
        x = tf.nn.dropout(x, rate=0.1)

        if return_attention:
            attn_output, attn_weights = self.self_attn(x, return_attention=True)
        else:
            attn_output = self.self_attn(x)
            attn_weights = None

        out = self.layer_norm(x + attn_output)
        pooled = self.pool(out)
        logits = self.out(pooled)

        if return_attention:
            return logits, attn_weights
        return logits

vocab_size = loader.target_embeddings.shape[0]
model = SelfAttentionWordPredictor(
    loader.embedding_layer, vocab_size, loader.embedding_dim,
    seq_len=WINDOW, num_heads=4
)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.build((None, WINDOW))
model.summary()

batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(min(len(X_train), 2000)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

callbacks = [keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)]

history = model.fit(train_ds, validation_data=val_ds, epochs=100, verbose=1, callbacks=callbacks)

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy (Self-Attention, window={WINDOW}): {acc:.3f}')

def predict_next_word(context_words, top_n=5):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    if len(context_ids) < WINDOW:
        context_ids = [0] * (WINDOW - len(context_ids)) + context_ids
    context_ids = np.array([context_ids[-WINDOW:]], dtype=np.int32)
    probs = model.predict(context_ids, verbose=0)[0]
    top_indices = np.argsort(probs)[-top_n:][::-1]
    return [(loader.decode(idx), float(probs[idx])) for idx in top_indices]

sample_contexts = [
    ['london', 'bridge', 'river', 'city', 'center'],
    ['bank', 'of', 'england', 'is', 'located'],
    ['queen', 'of', 'england', 'lives', 'in']
]

for context in sample_contexts:
    preds = predict_next_word(context, top_n=5)
    if preds:
        print(f"Contexto {context} -> {preds}")

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title(f'Self-Attention (window={WINDOW}) - Training History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualización: Heatmap de Self-Attention para window=32
def plot_attention_heatmap(context_words, model):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    if len(context_ids) < WINDOW:
        context_ids = [0] * (WINDOW - len(context_ids)) + context_ids
    context_ids = np.array([context_ids[-WINDOW:]], dtype=np.int32)

    _, attn_avg = model(context_ids, return_attention=True)
    attn_matrix = attn_avg[0].numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    im = axes[0].imshow(attn_matrix, cmap='YlOrRd', vmin=0, vmax=attn_matrix.max())
    axes[0].set_title('Self-Attention Matrix (32x32)')
    axes[0].set_xlabel('Palabra atendida (j)')
    axes[0].set_ylabel('Palabra que atiende (i)')
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    # Distribución de atención (promedio por posición)
    attended = attn_matrix.sum(axis=0)
    attending = attn_matrix.sum(axis=1)
    positions = np.arange(WINDOW)
    axes[1].plot(positions, attended, 'o-', label='Atención recibida', color='coral')
    axes[1].plot(positions, attending, 's-', label='Atención emitida', color='steelblue')
    axes[1].set_xlabel('Posición en la secuencia')
    axes[1].set_ylabel('Suma de pesos de atención')
    axes[1].set_title('Distribución de atención por posición')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    pos_max_attended = np.argmax(attended)
    pos_max_attending = np.argmax(attending)
    print(f"Posición más atendida: {pos_max_attended}")
    print(f"Posición que más atiende: {pos_max_attending}")
    print(f"Diagonal promedio (auto-atención): {np.trace(attn_matrix) / WINDOW:.4f}")

# Usar primeras 32 palabras del dataset como contexto
sample_words = words[:WINDOW]
print("Contexto para visualización (primeras 32 palabras):")
print(' '.join(sample_words))
plot_attention_heatmap(sample_words, model)

In [ ]:
# Comparación contra notebook 19 (que intentó window=64 con RNN y falló)
print("""
Comparación: RNN vs Self-Attention con ventanas grandes
""")
print(f"{"Modelo":<30} {"Window":<10} {"Accuracy":<10}")
print("-" * 50)
print(f"{"19 - RNN + clipping + truncation":<30} {"64":<10} {"0.105":<10}")
print(f"{"22 - Self-Attention (actual)":<30} {f"{WINDOW}":<10} {f"{acc:.3f}":<10}")
print(f"{"21 - Self-Attention (window=5)":<30} {"5":<10} {"?":<10}")
print()
if acc > 0.105:
    print("✅ Self-attention supera a la RNN con ventana grande.")
    print(f"   Mejora: {acc - 0.105:+.3f} vs notebook 19")
else:
    print("❌ Self-attention no mejoró a RNN con ventana grande.")

## Comparación completa

| Notebook | Modelo | Window | Test Accuracy |
|----------|--------|:-----:|:------------:|
| 18 | RNN vanilla | 5 | 0.403 |
| 19 | RNN + gradient clipping | 64 | 0.105 |
| 20 | RNN + Bahdanau Attention | 5 | 0.575 |
| 21 | Self-Attention manual | 5 | ? |
| **22** | **Self-Attention manual** | **32** | **?** |

**Conclusión clave:** Si self-attention con window=32 supera el 0.403 de 18,
confirma que self-attention escala a ventanas grandes mientras que RNN no.